# 1. Kiểm tra GPU
Notebook này nên chạy trên Google Colab T4/A100.


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f'VRAM free: {free/1e9:.2f} GB / total: {total/1e9:.2f} GB')
else:
    print('NO GPU - chỉ nên dùng để setup hoặc test rất ngắn')


# 2. Mount Google Drive tùy chọn
Dùng nếu muốn cache model qua nhiều session.


In [ ]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'


# 3. Clone repo và cài dependencies


In [ ]:
%cd /content
!rm -rf Fast-VietTTS
!git clone https://github.com/baobui0709/Fast-VietTTS.git
%cd Fast-VietTTS

!pip install -q --upgrade pip
!pip install -q --index-url https://download.pytorch.org/whl/cu124 torch==2.6.0 torchaudio==2.6.0
!pip install -q numpy==1.26.4 librosa soundfile num2words tqdm huggingface_hub gradio
!pip install -q s3tokenizer omegaconf conformer safetensors
!pip uninstall -y -q torchvision torchtext || true

%cd /content
!rm -rf TienHiep-TTS
!git clone https://github.com/yingchunbin/TienHiep-TTS.git
%cd /content/Fast-VietTTS


# 4. Import helper chung


In [ ]:
import sys, os, time, torch, torchaudio
from IPython.display import Audio, display

viterbox_src = '/content/TienHiep-TTS'
if viterbox_src not in sys.path:
    sys.path.insert(0, viterbox_src)

from src.engine import FastVietTTS
from src.audio_utils import prepare_reference, get_audio_duration


# 5. Load model


In [ ]:
tts = FastVietTTS(device='cuda' if torch.cuda.is_available() else 'cpu', use_fp16=False, compile_model=False)
print('Model loaded')


# 6. Upload và tiền xử lý reference audio


In [ ]:
from google.colab import files
import librosa, matplotlib.pyplot as plt
uploaded = files.upload()
ref_path = next(iter(uploaded.keys()))
raw, sr = librosa.load(ref_path, sr=None, mono=True)
processed_path = prepare_reference(ref_path, output_path='/content/ref_processed.wav')
proc, psr = librosa.load(processed_path, sr=None, mono=True)

plt.figure(figsize=(12,3))
plt.subplot(1,2,1); plt.plot(raw); plt.title('Waveform trước xử lý')
plt.subplot(1,2,2); plt.plot(proc); plt.title('Waveform sau xử lý')
plt.show()
print('Processed:', processed_path)


# 7. Test nhiều thông số


In [ ]:
params = [
    {'exaggeration':0.4, 'cfg_weight':0.6},
    {'exaggeration':0.5, 'cfg_weight':0.5},
    {'exaggeration':0.7, 'cfg_weight':0.4},
]
results = []
text = 'Giọng nói này được tạo để so sánh các thông số voice cloning.'
for i, p in enumerate(params, 1):
    t0 = time.time()
    wav = tts.generate(text, audio_prompt=processed_path, language='vi', **p)
    elapsed = time.time() - t0
    out = f'/content/voiceclone_{i}.wav'
    tts.save_audio(wav, out)
    dur = wav.shape[-1] / 24000
    results.append({**p, 'file': out, 'duration': dur, 'elapsed': elapsed, 'rtf': elapsed/max(dur,1e-6)})
    print(i, results[-1])
    display(Audio(out))


# 8. Bảng so sánh


In [ ]:
import pandas as pd
pd.DataFrame(results)


# 9. Mẹo chọn giọng mẫu tốt
- 5–20 giây, ít nhiễu.
- Nói rõ, không nhạc nền.
- Cùng ngôn ngữ với text cần đọc.
- Không quá nhiều người nói trong cùng file.

Tóm tắt: VoiceClone notebook hoàn tất.
